# PROVIDER - Bronze Layer

This notebook ingests the MIMIC-IV `provider` source table and stores it as a Bronze Delta table.

A small source check is completed before ingestion to confirm the table structure and provider identifier.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"

SOURCE_PATH = "s3://mimic-iii-demo-bucket/mimic-iv-dataset-demo-2.2-folder/hosp/provider.csv"

provider = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

print(f"Rows: {provider.count():,}")
print(f"Columns: {len(provider.columns)}")

provider.printSchema()


### Source Check

`PROVIDER` contains the de-identified provider identifiers used across the hospital data.

Before creating the Bronze table, `provider_id` is checked for missing or duplicate values.

In [0]:
# Check provider_id
total = provider.count()
unique_ids = provider.select("provider_id").distinct().count()
null_ids = provider.filter(F.col("provider_id").isNull()).count()

print(f"Rows: {total:,}")
print(f"Unique provider_id: {unique_ids:,}")
print(f"Null provider_id: {null_ids:,}")
print(f"Duplicate provider_id: {total - unique_ids:,}")

### Bronze Ingestion

`provider_id` is complete and unique across the source table.

The table is ready to be stored in Bronze with the original structure preserved and basic ingestion metadata added.

In [0]:
# Add ingestion metadata
provider_bronze = (
    provider
    .withColumn("_source_file", F.lit("provider.csv"))
    .withColumn("_source_module", F.lit("hosp"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as Delta table
(
    provider_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.provider_raw")
)

print("Created: workspace.bronze.provider_raw")

In [0]:
source_count = provider.count()
bronze_count = spark.table("workspace.bronze.provider_raw").count()

print(f"Source: {source_count:,}")
print(f"Bronze: {bronze_count:,}")
print(f"Match: {source_count == bronze_count}")

### Bronze Result

`PROVIDER` was successfully written to the Bronze layer with all source records preserved.

The `provider_id` field is complete and unique, and the Bronze table matches the source row count.